In [20]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import cred
import streamlit as st
import pandas as pd
import re
import requests


In [21]:

scope = "user-read-recently-played"
scope = "user-library-read"

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=cred.client_id, client_secret= cred.client_secret, redirect_uri=cred.redirect_url, scope=scope))

album_url = "4aawyAB9vmqN3uQ7FjRGTy"

# st.text_input(label="Please input the album's url:")
# st.write(type(album_url))
# st.write(sp.album(album_url))


# try:
#     type(album_url) is str
# except:
#     st.error('You need to enter a url')
#     st.stop()


album = sp.album(album_url)
album_upc = album['external_ids']['upc']
album_artist = album['artists'][0]['name']
album_title = album['name']
album_track_number = album['total_tracks']
# st.write(album_upc,'-',album_artist,'-', album_title, '-', album_track_number)

tracks_list = []

for i in range(album_track_number):
    track_number = album['tracks']['items'][i]['track_number']
    track_name = album['tracks']['items'][i]['name']
    track_artists = album['tracks']['items'][i]['artists'][0]['name']
    track_url = album['tracks']['items'][i]['external_urls']['spotify']
    # st.write(track_number, '-', track_name, '-', track_artists, '-', track_url)
    
    tracks_list.append({
                "Track Number": track_number,
                "Title": track_name,
                "Artist": track_artists,
                "URL": track_url
            })

        # Créer le DataFrame à partir de la liste de dictionnaires
df_tracks = pd.DataFrame(tracks_list)
# st.dataframe(df_tracks, use_container_width=True)
print(df_tracks)

    Track Number                                              Title   Artist  \
0              1                     Global Warming (feat. Sensato)  Pitbull   
1              2                   Don't Stop the Party (feat. TJR)  Pitbull   
2              3        Feel This Moment (feat. Christina Aguilera)  Pitbull   
3              4        Back in Time - featured in "Men In Black 3"  Pitbull   
4              5             Hope We Meet Again (feat. Chris Brown)  Pitbull   
5              6          Party Ain't Over (feat. Usher & Afrojack)  Pitbull   
6              7       Drinks for You (Ladies Anthem) (feat. J. Lo)  Pitbull   
7              8        Have Some Fun (feat. The Wanted & Afrojack)  Pitbull   
8              9                 Outta Nowhere (feat. Danny Mercer)  Pitbull   
9             10            Tchu Tchu Tcha (feat. Enrique Iglesias)  Pitbull   
10            11         Last Night (feat. Afrojack & Havana Brown)  Pitbull   
11            12                        

In [22]:
#Spotify API
# Get User's Top items

scope = "user-top-read"
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=cred.client_id, client_secret= cred.client_secret, redirect_uri=cred.redirect_url
                                                , scope=scope
                                              ))

#Can choose between long / medium / short term
time_range = 'medium_term'

top_tracks = sp.current_user_top_tracks(limit=5, offset=0, time_range=time_range)

track_details = []
track_number = len(top_tracks["items"])
track_ids = []
print(track_number)

for i in range(track_number):
    track_name = top_tracks['items'][i]['name']
    track_artists = top_tracks['items'][i]['artists'][0]['name']
    track_url = top_tracks['items'][i]['external_urls']['spotify']
    track_id = top_tracks['items'][i]['id']
    
    track_details.append({
                "Title": track_name,
                "Artist": track_artists,
                #"URL": track_url,
                "ID" : track_id
            })
    track_ids.append(track_id)

df_tracks = pd.DataFrame(track_details)
# st.dataframe(df_tracks, use_container_width=True)
print(df_tracks)
print(track_ids)




5
       Title         Artist                      ID
0      Woman            Ayọ  5dfgpp4s7V3W2AWqKaWxdL
1        DNA  Flavia Coelho  2F6bxE9v4Ml88iC3RY6Pd6
2     Closer            Ayọ  2dLDhUut13iRPz0YX6Vzma
3  Beautiful            Ayọ  5IMx2ejtEMJAVUnTw3KWZX
4    Paraíso  Flavia Coelho  5TcuMW8C0hHXMCWytzCcNI
['5dfgpp4s7V3W2AWqKaWxdL', '2F6bxE9v4Ml88iC3RY6Pd6', '2dLDhUut13iRPz0YX6Vzma', '5IMx2ejtEMJAVUnTw3KWZX', '5TcuMW8C0hHXMCWytzCcNI']


In [23]:
#Reccobeats
#Get Audio features

url = "https://api.reccobeats.com/v1/audio-features"

#Sound of silence - Simon & Garfunkel
track_ids = "5y788ya4NvwhBznoDIcXwK"
params = {
    'ids': track_ids
}

headers = {
  'Accept': 'application/json'
}

response = requests.get(url, params=params
#                         , headers=headers 
                       )

if response.status_code == 200:
    print(response.json())
else:
    print(f"Erreur : {response.status_code}")
    print(response.text)

{'content': [{'id': 'd62264f9-9f6e-4bf1-a635-b8fbf27bdde7', 'acousticness': 0.837, 'danceability': 0.525, 'energy': 0.216, 'instrumentalness': 0.0, 'liveness': 0.107, 'loudness': -13.551, 'speechiness': 0.0301, 'tempo': 106.761, 'valence': 0.328}]}


In [24]:
# Reccobeats 
# Get recommendations
url = "https://api.reccobeats.com/v1/track/recommendation"


#features
# danceability = st.slider("Danceability :", 0.0, 1.0, 0.5)

#0 to 1
acousticness = 0.5
# danceability
# energy
# instrumentalness
# liveness
# speechiness
# valence

# 0 (Major) or 1 (Minor)
# mode

# -1 to 11
# key

# -60 to 2 (dB)
# loudness

# 0 to 250 (BPM)
tempo = 120

# 0 to 100
popularity = 90

#Use features and top tracks of the user to create recommendations params
recommended_track_params = {}


recommended_track_params = {
    'size' : 1, 
    'seeds': track_ids,
    'acousticness' : acousticness,
    'tempo' : tempo,
    'popularity' : popularity
}

headers = {
  'Accept': 'application/json'
}

response = requests.get(url, params=recommended_track_params
#                         , headers=headers 
                       )

recommended_track = response.json()
recommended_track_name = recommended_track['content'][0]['trackTitle']
recommended_track_artist = recommended_track['content'][0]['artists'][0]['name']
recommended_track_spotify_url = recommended_track['content'][0]['href']


if response.status_code == 200:
    #print(recommended_track)
    print(recommended_track_name)
    print(recommended_track_artist)
    print(recommended_track_spotify_url)
else:
    print(f"Erreur : {response.status_code}")
    print(response.text)

Everybody Wants To Rule The World
Tears For Fears
https://open.spotify.com/track/4RvWPyQ5RL0ao9LPZeSouE


In [25]:
#Spotify API
# Save track for current user (in liked song playlist)

scope = "user-library-modify"
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=cred.client_id, client_secret= cred.client_secret, redirect_uri=cred.redirect_url
                                                , scope=scope
                                              ))

track_to_save = recommended_track_spotify_url
track_to_save_id = [re.split("/", track_to_save)[-1]]
print(track_to_save)
print(track_to_save_id)

sp.current_user_saved_tracks_add(tracks=track_to_save_id)




https://open.spotify.com/track/4RvWPyQ5RL0ao9LPZeSouE
['4RvWPyQ5RL0ao9LPZeSouE']


In [17]:
# Main script
# Get user top tracks


def api_spotify_auth(scope):
    sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=cred.client_id, client_secret= cred.client_secret, redirect_uri=cred.redirect_url, scope=scope))
    return sp

def get_top_tracks(time_range, track_nb, offset, sp):

    # API result
    top_tracks = sp.current_user_top_tracks(limit=track_nb, offset=offset, time_range=time_range)
    
    # Create dataframe and list of ids
    track_details = []
    track_number = len(top_tracks["items"])
    track_ids = []
    print(track_number)

    for i in range(track_number):
        track_name = top_tracks['items'][i]['name']
        track_artists = top_tracks['items'][i]['artists'][0]['name']
        track_url = top_tracks['items'][i]['external_urls']['spotify']
        track_id = top_tracks['items'][i]['id']

        track_details.append({
                    "Title": track_name,
                    "Artist": track_artists,
                    "URL": track_url
                    #"ID" : track_id
                })
        track_ids.append(track_id)

    df_tracks = pd.DataFrame(track_details)
    
    #replace by st.dataframe
    print(df_tracks)
    
    return track_ids


# widget to choose between long / medium / short term
time_range= st.pills("How long have you been listening to the tracks?", ['short_term', 'medium_term', 'long_term'])

# Other args
track_nb = 5
offset = 0
scope = "user-top-read"

sp = api_spotify_auth(scope)

track_ids = get_top_tracks(time_range, track_nb, offset, sp)

print(track_ids)



2025-08-07 15:05:51.670 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:05:51.670 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:05:51.671 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:05:51.672 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:05:51.673 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


5
       Title         Artist                                                URL
0      Woman            Ayọ  https://open.spotify.com/track/5dfgpp4s7V3W2AW...
1        DNA  Flavia Coelho  https://open.spotify.com/track/2F6bxE9v4Ml88iC...
2     Closer            Ayọ  https://open.spotify.com/track/2dLDhUut13iRPz0...
3  Beautiful            Ayọ  https://open.spotify.com/track/5IMx2ejtEMJAVUn...
4    Paraíso  Flavia Coelho  https://open.spotify.com/track/5TcuMW8C0hHXMCW...
['5dfgpp4s7V3W2AWqKaWxdL', '2F6bxE9v4Ml88iC3RY6Pd6', '2dLDhUut13iRPz0YX6Vzma', '5IMx2ejtEMJAVUnTw3KWZX', '5TcuMW8C0hHXMCWytzCcNI']


In [32]:
# Reccobeats 
# Get recommendations

def build_params(size, seeds):
    
    #Initialize params dico
    recommended_track_params = {
        'size' : size, 
        'seeds': seeds,
    }
    
    # Define param for each feature with widgets
    #0 to 1
    enable_sliders = st.toggle("Pimp track features")
    if enable_sliders:
        acousticness = st.slider("Acousticness :", 0.0, 1.0, 0.2)
        danceability = st.slider("Danceability :", 0.0, 1.0, 0.2)
        energy = st.slider("Energy :", 0.0, 1.0, 0.2)
        instrumentalness = st.slider("Instrumentalness :", 0.0, 1.0, 0.2)
        liveness = st.slider("Liveness :", 0.0, 1.0, 0.2)
        speechiness = st.slider("Speechiness :", 0.0, 1.0, 0.2)
        valence = st.slider("Valence :", 0.0, 1.0, 0.2)
    else:
        acousticness = None
        danceability = None
        energy = None
        instrumentalness = None
        liveness = None
        speechiness = None
        valence = None

    # 1 (Major) or 0 (Minor)
    options = ["Whatever", "Major", "Minor" ]
    mode = st.pills("Mode", options)
    mode_mapping ={
        "Whatever": None,
        "Major": 1,
        "Minor":0
    }
    mode = mode_mapping.get(mode)

    # -1 to 11
    options = ["Whatever","C", "C♯/D♭", "D", "D♯/E♭", "E", "F", "F♯/G♭", "G", "G♯/A♭", "A", "A♯/B♭", "B"]
    key = st.pills("Key", options)
    pitch_class_notation = {
        "Whatever" : None,
        "C": 0,
        "C♯/D♭": 1,
        "D": 2,
        "D♯/E♭": 3,
        "E": 4,
        "F": 5,
        "F♯/G♭": 6,
        "G": 7,
        "G♯/A♭": 8,
        "A": 9,
        "A♯/B♭": 10,
        "B": 11
    }
    key = pitch_class_notation.get(key)

    # 0 to 250 (BPM)
    enable_tempo = st.toggle("Pimp track tempo")
    if enable_tempo:
        tempo = st.slider("Tempo :", 1, 250, 1)
    else:
        tempo = None

    # 0 to 100
    popularity = st.slider("Popularity :", 1, 100, 1)
    
    # Build params dico
    all_params = {
        'acousticness': acousticness,
        'danceability': danceability,
        'energy': energy,
        'instrumentalness': instrumentalness,
        'liveness': liveness,
        'speechiness': speechiness,
        'valence': valence,
        'mode': mode,
        'key': key,
        'tempo': tempo,
        'popularity': popularity
    }

    # Add params to final dico where value is not None
    for param_name, param_value in all_params.items():
        if param_value is not None:
            recommended_track_params[param_name] = param_value
    
    return recommended_track_params


def get_recommendation(url, recommended_track_params):
    # Get API response
    response = requests.get(url, params=recommended_track_params)
    recommended_track = response.json()
    
    recommended_track_number = len(recommended_track["content"])
    # st.write(recommended_track_number)

    # Create dataframe with recommended tracks
    recommended_track_details = []
    for i in range(recommended_track_number):
        recommended_track_name = recommended_track['content'][i]['trackTitle']
        recommended_track_artist = recommended_track['content'][i]['artists'][0]['name']
        recommended_track_spotify_url = recommended_track['content'][i]['href']
        recommended_track_details.append({
                        "Title": recommended_track_name,
                        "Artist": recommended_track_artist,
                        "URL": recommended_track_spotify_url
                        #"ID" : track_id
                    })
    return recommended_track_details

# Can be 1 to 100
size = 10
# size = st.number_input("Insert a number of track")

# Defined in previous functions - Spotify API
seeds = track_ids
    
recommended_track_params = build_params(size, seeds)
url = "https://api.reccobeats.com/v1/track/recommendation"
recommendation = get_recommendation(url, recommended_track_params)
print(pd.DataFrame(recommendation))

2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 15:12:24.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

                               Title                  Artist  \
0        Ik Wil Niet Trouwen Met Jou           Corry Konings   
1           Ja Dil Tenoon De Chhadia   Nusrat Fateh Ali Khan   
2     Self-Generation As Vajrayogini  Lama Gangchen Rinpoche   
3  Jesteś w stronie nieznanej (Live)         Grzegorz Turnau   
4        Mandolin Man and His Secret                 Donovan   
5                   Ceux qui passent                  Aliose   
6                    Lucifer's Dance           Jennie Ståbis   
7            Mély erdőn ibolya-virág   La La Land Gyerekzene   
8                              Ho Lo                Phạm Duy   

                                                 URL  
0  https://open.spotify.com/track/1urH80sMaZW5SEO...  
1  https://open.spotify.com/track/5lVlkr9z0mpzvCh...  
2  https://open.spotify.com/track/4XysYSNrguoZZWU...  
3  https://open.spotify.com/track/3uLtRxo4g2Oglm8...  
4  https://open.spotify.com/track/78PN4xjrIToY58a...  
5  https://open.spotify.com/t

In [42]:
#Spotify API
# Save track for current user (in liked song playlist)

def save_track_to_liked_songs(recommendation):
    # Authenticate to API with the appropriate scope
    sp = api_spotify_auth("user-library-modify")

    # Retrieve Ids from Spotify URLS in recommendation dico
    recommended_track_spotify_id = []
    track_number = len(recommendation)
    for i in range(track_number):
        track_spotify_URL = recommendation[i]['URL']
        track_spotify_id = re.split("/", track_spotify_URL)[-1]
        recommended_track_spotify_id.append(track_spotify_id)
    response = sp.current_user_saved_tracks_add(tracks=recommended_track_spotify_id)
    return response


saving_tracks = save_track_to_liked_songs(recommendation)
if saving_tracks == None:
    print("OK")

OK
